# ZTLF 00 — Reproducible environment setup

Portable rework of the original study — Colab + PySpark + Delta instead of Azure Databricks.

This sets up a pinned Spark/Delta environment and the project folder structure in Drive. Run it once per session, before anything else.

Why bother pinning versions: last time I couldn't reproduce my own results a few weeks later because I never wrote down which Databricks runtime or library versions I'd used. Everything here gets pinned and dumped into `environment_manifest.json` so I (or anyone else) can rebuild the same stack later.

One thing to watch: `delta-spark` has to match the PySpark major.minor version or it just won't import. The pairing below is what I've actually tested. Colab sometimes changes its default JDK on you between sessions, so the Java install step pins that explicitly instead of trusting whatever happens to be there.


In [ ]:
#@title Pinned dependency versions { display-mode: "form" }
# Single source of truth for the whole project. Change here, nowhere else.

PYSPARK_VERSION   = "3.5.3"
DELTA_VERSION     = "3.2.1"     # delta-spark 3.2.x pairs with PySpark 3.5.x
PANDAS_VERSION    = "2.2.3"
NUMPY_VERSION     = "1.26.4"
SKLEARN_VERSION   = "1.5.2"
MATPLOTLIB_VERSION= "3.9.2"

RANDOM_SEED = 20260803   # fixed project-wide seed; never change after first run

print("PySpark", PYSPARK_VERSION, "| Delta", DELTA_VERSION, "| seed", RANDOM_SEED)

In [ ]:
#@title Install Java 11 and pinned Python packages
# Delta 3.x + Spark 3.5 run on Java 11 or 17. We install 11 explicitly so the
# environment does not silently change when Colab updates its base image.
import subprocess, sys, os

def sh(cmd):
    print("$", cmd)
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-3000:]); print(r.stderr[-3000:])
        raise RuntimeError(f"failed: {cmd}")
    return r.stdout

sh("apt-get -qq update")
sh("apt-get -qq install -y openjdk-11-jdk-headless > /dev/null")

# Locate the JDK rather than hardcoding an arch-specific path
import glob
cands = sorted(glob.glob("/usr/lib/jvm/java-11-openjdk*"))
assert cands, "Java 11 not found after install"
os.environ["JAVA_HOME"] = cands[0]
print("JAVA_HOME =", os.environ["JAVA_HOME"])

sh(f"pip -q install pyspark=={PYSPARK_VERSION} delta-spark=={DELTA_VERSION} "
   f"pandas=={PANDAS_VERSION} numpy=={NUMPY_VERSION} "
   f"scikit-learn=={SKLEARN_VERSION} matplotlib=={MATPLOTLIB_VERSION}")
print("\ninstall complete")

In [ ]:
#@title Mount Google Drive and create the project tree
from google.colab import drive
drive.mount('/content/drive')

import pathlib

# ---- EDIT THIS if your folder is named differently -------------------------
PROJECT_ROOT = pathlib.Path("/content/drive/MyDrive/Paper1")
# ---------------------------------------------------------------------------

SUBDIRS = ["data/raw", "data/lakehouse", "src", "outputs/tables",
           "outputs/figures", "outputs/metrics", "outputs/logs", "notebooks"]

for d in SUBDIRS:
    (PROJECT_ROOT / d).mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
for d in SUBDIRS:
    print("  ", d)

In [ ]:
#@title Start Spark with Delta Lake
import os, sys
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

# Colab is a single node. We configure for correctness and determinism,
# NOT for scale -- the paper must not claim production throughput.
builder = (
    SparkSession.builder
    .appName("ZTLF")
    .master("local[*]")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog",
            "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.sql.session.timeZone", "UTC")          # determinism
    .config("spark.sql.shuffle.partitions", "8")          # determinism
    .config("spark.databricks.delta.retentionDurationCheck.enabled", "false")
    .config("spark.driver.memory", "8g")
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

print("Spark", spark.version)
print("Delta extension loaded:",
      "DeltaSparkSessionExtension" in spark.conf.get("spark.sql.extensions"))

In [ ]:
#@title Verify Delta round-trip (fail fast if the stack is broken)
test_path = str(PROJECT_ROOT / "data/lakehouse/_env_check")

df = spark.createDataFrame([(1, "a"), (2, "b")], ["id", "val"])
df.write.format("delta").mode("overwrite").save(test_path)

back = spark.read.format("delta").load(test_path)
assert back.count() == 2, "Delta round-trip failed"

from delta.tables import DeltaTable
hist = DeltaTable.forPath(spark, test_path).history()
print("Delta round-trip OK. Table history rows:", hist.count())
hist.select("version", "operation").show(truncate=False)

In [ ]:
#@title Write the environment manifest (goes into the reproducibility package)
import json, platform, datetime, subprocess

manifest = {
    "project": "ZTLF portable rework",
    "generated_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "random_seed": RANDOM_SEED,
    "python": sys.version,
    "platform": platform.platform(),
    "java_home": os.environ.get("JAVA_HOME"),
    "java_version": subprocess.run(
        ["java", "-version"], capture_output=True, text=True).stderr.strip(),
    "spark_version": spark.version,
    "pinned": {
        "pyspark": PYSPARK_VERSION, "delta-spark": DELTA_VERSION,
        "pandas": PANDAS_VERSION, "numpy": NUMPY_VERSION,
        "scikit-learn": SKLEARN_VERSION, "matplotlib": MATPLOTLIB_VERSION,
    },
    "spark_conf": {k: v for k, v in sorted(spark.sparkContext.getConf().getAll())
                   if k.startswith(("spark.sql", "spark.driver", "spark.databricks"))},
}

out = PROJECT_ROOT / "outputs/logs/environment_manifest.json"
out.write_text(json.dumps(manifest, indent=2))
print("wrote", out)
print(json.dumps(manifest["pinned"], indent=2))

---
### Next
Environment's ready — move on to ZTLF 01 (ingestion + natural-defect census).

Before that, drop these into `Paper1/data/raw/`:

| File | Source | License |
|---|---|---|
| `bank.csv`, `bank-full.csv` | UCI Bank Marketing (id 222) | CC BY 4.0 |
| `diabetic_data.csv` | UCI Diabetes 130-US Hospitals (id 296) | CC BY 4.0 |
| `online_retail_II.csv` | UCI Online Retail II (id 502) | non-commercial only, don't redistribute |
